In [5]:
# ==========================
# 1. Imports e constantes
# ==========================

import csv
import platform
import re
from pathlib import Path

import duckdb
import pandas as pd

COLUNAS = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "TRACTORA", "REMOLQUE",
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT", "PESO_BRUTO",
    "CODEUT", "ESTADO_UT", "RANGO_UT", "FCARGA", "ACTIVIDAD",
    "CODEDT", "ESTADO_DT", "REFERENCIA", "CODACT", "LOCORIGEN",
    "PROV_ORIGEN", "PAISORIGEN", "CPOSTAL", "LOCDESTINO", "PROV_DESTINO",
    "PAISDESTINO", "CPOSTAD", "KM", "FENTREGA", "ORIGEN",
    "ENTREGAR", "PROV_ENTREGAR", "PAISENTREGAR", "DESTINO", "PALETS",
    "PREFAC", "RUTA", "COBROREAL", "GESTION", "DEPART",
    "USCODE", "USUARIO", "TIPOCLIENTE", "TIPOFLUJO", "WMSCODRGT",
    "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TEMP_MERC_PED",
    "TIPOPALETA", "CAMION_TIPO", "CAMION_CAPACIDAD", "TIPO_COMBUSTIBLE",
    "KMREALES", "ALBARAN"
]

COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
REGEX_ANO = re.compile(r"^(\d{4})")


In [6]:
# ==========================
# 2. Configuração
# ==========================

if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00_DB\2026.duckdb"
    )
    PASTA_FICHEIROS = Path(
        r"C:\Users\LISARR\Documents\python\000.Dados_input\inform_27"
    )

elif platform.system() == "Darwin":
    DB_PATH = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb"
    )
    PASTA_FICHEIROS = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados_output"
    )

else:
    raise OSError(f"Sistema operativo não suportado: {platform.system()}")

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DB_PATH)) as con:
    con.execute("SELECT 1")

print(f"✓ BD: {DB_PATH}")
print(f"✓ Dados: {PASTA_FICHEIROS}")


✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb
✓ Dados: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados_output


In [10]:
# ==========================
# 3. Funções
# ==========================

def obter_ano(valor):
    if valor is None:
        return None

    texto = str(valor).strip()
    match = REGEX_ANO.match(texto)

    if not match or match.group(1) == "0000":
        return None

    return int(match.group(1))


def tabela_ano(ano):
    return f"inform_27_{ano}"


def criar_tabela(con, ano):
    tabela = tabela_ano(ano)
    colunas_sql = ", ".join(
        f'"{coluna}" VARCHAR'
        for coluna in COLUNAS
    )

    con.execute(f"""
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id BIGINT PRIMARY KEY,
            {colunas_sql},
            "{COLUNA_ORIGEM}" VARCHAR
        )
    """)

    con.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}" '
        f'ON "{tabela}" ("CODEDT", "FENTREGA", "CODEUT")'
    )

    return tabela


def preparar_dataframe(linhas, id_inicial):
    colunas = COLUNAS + [COLUNA_ORIGEM]

    df_batch = pd.DataFrame(
        linhas,
        columns=colunas,
        dtype=object,
    )

    df_batch.insert(
        0,
        "id",
        range(id_inicial, id_inicial + len(df_batch)),
    )

    return df_batch


def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)

    antes = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    max_id = con.execute(
        f'SELECT COALESCE(MAX(id), 0) FROM "{tabela}"'
    ).fetchone()[0]

    df_batch = preparar_dataframe(
        linhas,
        max_id + 1,
    )

    con.register(
        "batch_inform_27",
        df_batch,
    )

    try:
        con.execute(f"""
            INSERT OR IGNORE INTO "{tabela}"
            SELECT *
            FROM batch_inform_27
        """)

    finally:
        con.unregister("batch_inform_27")

    depois = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    inseridos = depois - antes
    duplicados = len(linhas) - inseridos

    return inseridos, duplicados


In [11]:
# ==========================
# 4. Importação
# ==========================

ficheiros = sorted(
    PASTA_FICHEIROS.rglob("*.csv")
)

if not ficheiros:
    raise FileNotFoundError(
        f"Nenhum ficheiro .csv em: {PASTA_FICHEIROS}"
    )

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "erros": 0,
}

with duckdb.connect(str(DB_PATH)) as con:

    for numero, caminho in enumerate(
        ficheiros,
        1,
    ):
        transacao_aberta = False

        try:
            # ==========================
            # 4.1. Ler CSV
            # ==========================

            with caminho.open(
                "r",
                encoding="utf-8",
                newline="",
            ) as f:
                reader = csv.reader(
                    f,
                    delimiter=";",
                )

                cabecalho = next(reader)
                linhas = list(reader)

            if COLUNA_DATA not in cabecalho:
                print(
                    f"[{numero}/{len(ficheiros)}] "
                    f"{caminho.name} — sem {COLUNA_DATA}"
                )

                totais["erros"] += 1
                continue

            mapa_indices = {
                coluna: indice
                for indice, coluna in enumerate(cabecalho)
            }

            indice_data = mapa_indices[COLUNA_DATA]

            linhas_por_ano = {}
            sem_data = 0

            # ==========================
            # 4.2. Preparar linhas
            # ==========================

            for valores in linhas:

                valor_data = (
                    valores[indice_data]
                    if indice_data < len(valores)
                    else None
                )

                ano = obter_ano(valor_data)

                if ano is None:
                    sem_data += 1
                    continue

                linha = [
                    valores[mapa_indices[coluna]]
                    if (
                        coluna in mapa_indices
                        and mapa_indices[coluna] < len(valores)
                    )
                    else None
                    for coluna in COLUNAS
                ]

                linha.append(caminho.name)

                linhas_por_ano.setdefault(
                    ano,
                    [],
                ).append(linha)

            # ==========================
            # 4.3. Inserir no DuckDB
            # ==========================

            con.begin()
            transacao_aberta = True

            novos_ficheiro = 0
            duplicados_ficheiro = 0
            inseridos_ficheiro = {}

            for ano, batch in sorted(
                linhas_por_ano.items()
            ):
                tabela = tabela_ano(ano)

                # ==========================
                # 4.3.1. Garantir tabela
                # ==========================

                criar_tabela(
                    con,
                    ano,
                )

                antes = con.execute(
                    f'SELECT COUNT(*) FROM "{tabela}"'
                ).fetchone()[0]

                # ==========================
                # 4.3.2. Próximo ID
                # ==========================

                max_id = con.execute(
                    f"""
                    SELECT COALESCE(
                        MAX(TRY_CAST(id AS BIGINT)),
                        0
                    )
                    FROM "{tabela}"
                    """
                ).fetchone()[0]

                # ==========================
                # 4.3.3. DataFrame
                # ==========================

                df_batch = preparar_dataframe(
                    batch,
                    int(max_id) + 1,
                )

                con.register(
                    "batch_inform_27",
                    df_batch,
                )

                try:
                    con.execute(
                        f"""
                        INSERT OR IGNORE INTO "{tabela}"
                        SELECT *
                        FROM batch_inform_27
                        """
                    )

                finally:
                    con.unregister(
                        "batch_inform_27"
                    )

                # ==========================
                # 4.3.4. Resultado
                # ==========================

                depois = con.execute(
                    f'SELECT COUNT(*) FROM "{tabela}"'
                ).fetchone()[0]

                inseridos = depois - antes
                duplicados = len(batch) - inseridos

                inseridos_ficheiro[ano] = inseridos
                novos_ficheiro += inseridos
                duplicados_ficheiro += duplicados

            con.commit()
            transacao_aberta = False

            # ==========================
            # 4.4. Totais
            # ==========================

            totais["ficheiros"] += 1
            totais["linhas"] += len(linhas)
            totais["duplicados"] += duplicados_ficheiro
            totais["sem_data"] += sem_data

            for ano, inseridos in inseridos_ficheiro.items():

                totais["inseridos"][ano] = (
                    totais["inseridos"].get(
                        ano,
                        0,
                    )
                    + inseridos
                )

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} | "
                f"+{novos_ficheiro:,} novos | "
                f"{duplicados_ficheiro:,} duplicados | "
                f"{sem_data:,} sem data"
            )

        except Exception as erro:

            if transacao_aberta:
                con.rollback()

            totais["erros"] += 1

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} — ERRO: {erro}"
            )

    con.checkpoint()


# ==========================
# 5. Resumo
# ==========================

print("\n--- RESUMO ---")

print(
    f"Ficheiros: {totais['ficheiros']} | "
    f"Linhas lidas: {totais['linhas']:,} | "
    f"Duplicadas: {totais['duplicados']:,} | "
    f"Sem data: {totais['sem_data']:,} | "
    f"Erros: {totais['erros']}"
)

for ano, quantidade in sorted(
    totais["inseridos"].items()
):

    print(
        f"Linhas novas em {ano}: "
        f"{quantidade:,}"
    )

[1/1] SAL_DAT027.csv | +5,154 novos | 1,514 duplicados | 0 sem data

--- RESUMO ---
Ficheiros: 1 | Linhas lidas: 6,668 | Duplicadas: 1,514 | Sem data: 0 | Erros: 0
Linhas novas em 2026: 5,154


In [12]:
# ==========================
# 6. Validação final
# ==========================

with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    tabelas = con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name LIKE 'inform_27_%'
        ORDER BY table_name
    """).fetchall()

    validacao = []

    for (tabela,) in tabelas:
        total, ids_unicos = con.execute(f"""
            SELECT
                COUNT(*) AS total,
                COUNT(DISTINCT id) AS ids_unicos
            FROM "{tabela}"
        """).fetchone()

        duplicados_chave = con.execute(f"""
            SELECT COUNT(*)
            FROM (
                SELECT
                    "CODEDT",
                    "FENTREGA",
                    "CODEUT",
                    COUNT(*) AS n
                FROM "{tabela}"
                GROUP BY
                    "CODEDT",
                    "FENTREGA",
                    "CODEUT"
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

        validacao.append({
            "Tabela": tabela,
            "Linhas": total,
            "IDs_Unicos": ids_unicos,
            "Chaves_Duplicadas": duplicados_chave,
            "OK": (
                total == ids_unicos
                and duplicados_chave == 0
            ),
        })

df_validacao = pd.DataFrame(validacao)

df_validacao


,Tabela,Linhas,IDs_Unicos,Chaves_Duplicadas,OK
0,inform_27_2026,779393,779393,0,True
